# Spike QC — Local Debug (Simplified)

This is a **local-first**, reduced-scope notebook for debugging pipeline issues faster.

Flow:
1. Enumerate SpikeForest rows from SHA1 manifests
2. Preflight + feature extraction (checkpointed)
3. Basic data audit
4. Grouped holdout model training (`fmiss`, `fpos`)


In [19]:
from pathlib import Path
import json
import time
import gc
import random
from datetime import datetime
from collections import defaultdict
import os
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

cwd = Path.cwd().resolve()
if cwd.name == "notebooks" and cwd.parent.name == "local_debug":
    WORK_ROOT = cwd.parent
elif (cwd / "local_debug").exists():
    WORK_ROOT = cwd / "local_debug"
elif (cwd.parent / "local_debug").exists():
    WORK_ROOT = cwd.parent / "local_debug"
else:
    raise RuntimeError(f"Could not locate local_debug folder from cwd={cwd}")

ARTIFACTS_DIR = WORK_ROOT / "artifacts"
CACHE_DIR = WORK_ROOT / "cache"
LOGS_DIR = WORK_ROOT / "logs"
DATA_DIR = WORK_ROOT / "data"
MODELS_DIR = ARTIFACTS_DIR / "models"
PARTS_DIR = ARTIFACTS_DIR / "train_parts"

for p in [ARTIFACTS_DIR, CACHE_DIR, LOGS_DIR, DATA_DIR, MODELS_DIR, PARTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

MANIFEST_JSON = ARTIFACTS_DIR / "manifest_selected.json"
PREFLIGHT_JSON = ARTIFACTS_DIR / "preflight_summary.json"
STATE_JSON = ARTIFACTS_DIR / "extract_state.json"
TRAIN_PARQUET = ARTIFACTS_DIR / "features_train.parquet"
AUDIT_JSON = ARTIFACTS_DIR / "audit_summary.json"
METRICS_JSON = ARTIFACTS_DIR / "holdout_metrics.json"
PREDICTIONS_CSV = ARTIFACTS_DIR / "holdout_predictions.csv"
RUN_LOG = LOGS_DIR / "run.log"

DEFAULT_OUTPUTS_URI = "sha1://789de61ef00d1ca94f4a2d43d75c3346bdfe0d0a?label=spikeforest-sorting-outputs.json"
DEFAULT_RECORDINGS_URI = "sha1://1d343ed7e876ffd73bd8e0daf3b8a2c4265b783c?spikeforest-recordings.json"
HYBRID_JANELIA_OUTPUTS_URI = "sha1://9259d3ec1d981560e35c2ca41e59c39f2af3d37e?label=spikeforest-sorting-outputs.json"
HYBRID_JANELIA_RECORDINGS_URI = "sha1://43298d72b2d0860ae45fc9b0864137a976cb76e8?hybrid-janelia-spikeforest-recordings.json"

STUDY_SETS = [
    "HYBRID_JANELIA",
    "PAIRED_ENGLISH",
    "PAIRED_BOYDEN",
    "PAIRED_MEA64C_YGER",
    "PAIRED_KAMPFF",
    "PAIRED_CRCNS_HC1",
]
ALLOWED_SORTERS = ["IronClust", "KiloSort2", "HerdingSpikes2", "KiloSort"]
MAX_ROWS = 260
INCLUDE_FAILED_SORTERS = False

print("WORK_ROOT:", WORK_ROOT)
print("MAX_ROWS:", MAX_ROWS)
print("STUDY_SETS:", STUDY_SETS)
print("ALLOWED_SORTERS:", ALLOWED_SORTERS)


WORK_ROOT: /Users/paulruiz/Documents/Predicting_Good_Units/local_debug
MAX_ROWS: 260
STUDY_SETS: ['HYBRID_JANELIA', 'PAIRED_ENGLISH', 'PAIRED_BOYDEN', 'PAIRED_MEA64C_YGER', 'PAIRED_KAMPFF', 'PAIRED_CRCNS_HC1']
ALLOWED_SORTERS: ['IronClust', 'KiloSort2', 'HerdingSpikes2', 'KiloSort']


In [20]:
import importlib
import os
import requests

# local timeout patch (simple + idempotent)
if not getattr(requests.sessions.Session.request, "_spike_qc_local_timeout_patch", False):
    _base_request = requests.sessions.Session.request
    def _patched_request(self, method, url, **kwargs):
        kwargs.setdefault("timeout", (30, 180))
        return _base_request(self, method, url, **kwargs)
    _patched_request._spike_qc_local_timeout_patch = True
    requests.sessions.Session.request = _patched_request

required = [
    "kachery_cloud",
    "spikeforest",
    "spikeinterface.core",
    "spikeinterface.comparison",
    "spikeinterface.extractors",
    "xgboost",
]

missing = []
for m in required:
    try:
        importlib.import_module(m)
    except Exception as exc:
        missing.append((m, str(exc)))

if missing:
    raise RuntimeError(f"Missing/failed imports: {missing}")

import kachery_cloud as kc

# Ensure kachery has a persistent local dir + client keys for SHA1 downloads
os.environ.setdefault("KACHERY_CLOUD_DIR", str((CACHE_DIR / ".kachery-cloud").resolve()))
Path(os.environ["KACHERY_CLOUD_DIR"]).mkdir(parents=True, exist_ok=True)

def ensure_kachery_client_keys():
    try:
        init_fn = getattr(kc, "init", None)
        if callable(init_fn):
            init_fn()
    except Exception:
        pass
    try:
        import kachery_cloud._client_keys as ck
        ck._get_client_keys_hex(generate_if_missing=True)
        pub, priv = ck._get_client_keys_hex(generate_if_missing=False)
        return (pub is not None) and (priv is not None)
    except Exception as exc:
        print("Client key bootstrap failed:", exc)
        return False

if not ensure_kachery_client_keys():
    raise RuntimeError(
        "kachery client keys are required and could not be created automatically. "
        "Check write permissions for KACHERY_CLOUD_DIR and rerun this cell."
    )
print("KACHERY_CLOUD_DIR:", os.environ["KACHERY_CLOUD_DIR"])

def pick(d, *keys):
    for k in keys:
        if isinstance(d, dict) and k in d and d[k] is not None:
            return d[k]
    return None

def to_int_or_none(v):
    try:
        return int(v)
    except Exception:
        return None

def infer_study_set(study_name, source_name):
    s = str(study_name or "").lower()
    if source_name == "hybrid_janelia":
        return "HYBRID_JANELIA"
    if "paired_english" in s:
        return "PAIRED_ENGLISH"
    if "paired_boyden" in s:
        return "PAIRED_BOYDEN"
    if "paired_mea64c" in s:
        return "PAIRED_MEA64C_YGER"
    if "paired_kampff" in s:
        return "PAIRED_KAMPFF"
    if "paired_crcns" in s or "paired_crcns_hc1" in s:
        return "PAIRED_CRCNS_HC1"
    if "hybrid_" in s:
        return "HYBRID_JANELIA"
    return None

def extract_list_payload(payload, kind, source_name):
    if payload is None:
        return [], {"shape": "none"}
    if isinstance(payload, list):
        return payload, {"shape": "list", "n": len(payload)}
    if not isinstance(payload, dict):
        raise RuntimeError(
            f"{kind} payload has unsupported type for source={source_name}: {type(payload)}"
        )

    keys = list(payload.keys())
    if kind == "outputs":
        candidates = [
            "sortingOutputs", "sorting_outputs", "sortingoutputs", "sortingResults",
            "sorting_results", "sortings", "results", "items", "data",
        ]
    else:
        candidates = [
            "recordings", "recordingList", "recording_list", "items", "data",
        ]

    for k in candidates:
        v = payload.get(k)
        if isinstance(v, list):
            return v, {"shape": f"dict[{k}]", "n": len(v), "keys": keys[:20]}
        if isinstance(v, dict):
            for kk in candidates + ["items", "list", "values"]:
                vv = v.get(kk)
                if isinstance(vv, list):
                    return vv, {"shape": f"dict[{k}][{kk}]", "n": len(vv), "keys": list(v.keys())[:20]}

    list_fields = [(k, v) for k, v in payload.items() if isinstance(v, list)]
    if len(list_fields) == 1:
        k, v = list_fields[0]
        return v, {"shape": f"dict[{k}]", "n": len(v), "keys": keys[:20]}

    if payload and all(isinstance(v, dict) for v in payload.values()):
        vals = list(payload.values())
        return vals, {"shape": "dict_values", "n": len(vals), "keys": keys[:20]}

    flat = []
    for v in payload.values():
        if isinstance(v, list):
            flat.extend([x for x in v if isinstance(x, dict)])
    if flat:
        return flat, {"shape": "flattened_list_values", "n": len(flat), "keys": keys[:20]}

    raise RuntimeError(
        f"Could not find row-list in {kind} payload for source={source_name}. "
        f"Top-level keys: {keys[:30]}"
    )

sources = [
    {
        "name": "default",
        "outputs_uri": DEFAULT_OUTPUTS_URI,
        "recordings_uri": DEFAULT_RECORDINGS_URI,
    },
    {
        "name": "hybrid_janelia",
        "outputs_uri": HYBRID_JANELIA_OUTPUTS_URI,
        "recordings_uri": HYBRID_JANELIA_RECORDINGS_URI,
    },
]

rows = []
raw_study_sets = []
raw_sorters = []
raw_counts = {}

allowed_sorters_norm = {s.lower() for s in ALLOWED_SORTERS}

for src in sources:
    src_name = src["name"]
    print(f"Loading source={src_name}")

    outputs_raw = kc.load_json(src["outputs_uri"])
    recordings_raw = kc.load_json(src["recordings_uri"])

    outputs, outputs_info = extract_list_payload(outputs_raw, "outputs", src_name)
    recordings, recordings_info = extract_list_payload(recordings_raw, "recordings", src_name)

    raw_counts[src_name] = {
        "outputs_raw_type": str(type(outputs_raw)),
        "recordings_raw_type": str(type(recordings_raw)),
        "outputs_parsed": outputs_info,
        "recordings_parsed": recordings_info,
    }

    rec_map = {}
    for r in recordings:
        study_name = pick(r, "studyName", "study_name", "study")
        recording_name = pick(r, "recordingName", "recording_name", "recording")
        study_set = pick(r, "studySetName", "study_set_name", "studySet", "study_set")
        if study_name and recording_name and study_set:
            rec_map[(str(study_name), str(recording_name))] = str(study_set)

    for o in outputs:
        study_name = pick(o, "studyName", "study_name", "study")
        recording_name = pick(o, "recordingName", "recording_name", "recording")
        sorter_name = pick(o, "sorterName", "sorter_name", "sorter")
        if not (study_name and recording_name and sorter_name):
            continue

        study_name = str(study_name)
        recording_name = str(recording_name)
        sorter_name = str(sorter_name)
        raw_sorters.append(sorter_name)

        study_set = pick(o, "studySetName", "study_set_name", "studySet", "study_set")
        if study_set is None:
            study_set = rec_map.get((study_name, recording_name))
        if study_set is None:
            study_set = infer_study_set(study_name, src_name)
        if study_set is None:
            continue
        study_set = str(study_set)
        raw_study_sets.append(study_set)

        if study_set not in STUDY_SETS:
            continue
        if sorter_name.lower() not in allowed_sorters_norm:
            continue

        timed_out = bool(pick(o, "timedOut", "timed_out") or False)
        return_code = to_int_or_none(pick(o, "returnCode", "return_code"))
        sorting_object = pick(o, "sortingObject", "sorting_object")

        if not INCLUDE_FAILED_SORTERS:
            if timed_out:
                continue
            if (return_code is not None) and (return_code != 0):
                continue
            if sorting_object is None:
                continue

        rows.append({
            "study_set": study_set,
            "study_name": study_name,
            "recording_name": recording_name,
            "sorter_name": sorter_name,
            "source_name": src_name,
            "source_outputs_uri": src["outputs_uri"],
            "source_recordings_uri": src["recordings_uri"],
            "return_code": return_code,
            "timed_out": timed_out,
            "sorting_object": sorting_object,
        })

seen = set()
dedup = []
for r in rows:
    k = (r["study_set"], r["study_name"], r["recording_name"], r["sorter_name"])
    if k in seen:
        continue
    seen.add(k)
    dedup.append(r)

rows = dedup[:MAX_ROWS]
manifest_df = pd.DataFrame(rows)
manifest_df.to_json(MANIFEST_JSON, orient="records", indent=2)

print("Raw loaded counts:", json.dumps(raw_counts, indent=2))
if raw_study_sets:
    print("Raw detected study_set (top):")
    print(pd.Series(raw_study_sets).value_counts().head(20))
if raw_sorters:
    print("Raw detected sorters (top):")
    print(pd.Series(raw_sorters).value_counts().head(20))

print(f"Selected rows: {len(manifest_df)}")
if not manifest_df.empty:
    print("By study_set:")
    print(manifest_df.groupby("study_set").size().sort_values(ascending=False))
    print("By sorter_name:")
    print(manifest_df.groupby("sorter_name").size().sort_values(ascending=False))
else:
    print("No rows selected with current filters.")
    print("Current STUDY_SETS:", STUDY_SETS)
    print("Current ALLOWED_SORTERS:", ALLOWED_SORTERS)
    print("Try broader settings:")
    print("- INCLUDE_FAILED_SORTERS = True")
    print("- ALLOWED_SORTERS = sorted(pd.Series(raw_sorters).dropna().astype(str).unique().tolist())")



This client has already been registered.
Click the following link to configure the client:
https://kachery-gateway.figurl.org/client/7309453d54ef4aa40c3a0fcf7bd26ad856c067c4b53c68a1a6cae6424c89cc68

Client ID: 7309453d54ef4aa40c3a0fcf7bd26ad856c067c4b53c68a1a6cae6424c89cc68
Label: MacBook-Pro-de-Paul-2.local
Owner: paulopasso

* Kachery-cloud is intended for collaborative sharing of data for scientific research. It should not be used for other purposes.
KACHERY_CLOUD_DIR: /Users/paulruiz/Documents/Predicting_Good_Units/local_debug/cache/.kachery-cloud
Loading source=default


RuntimeError: outputs is not a list for source=default: <class 'dict'>

In [ ]:
import spikeforest as sf
import spikeinterface.core as si
import spikeinterface.comparison as sc
import spikeinterface.extractors as sie

def log(msg):
    line = f"[{datetime.now().isoformat(timespec='seconds')}] {msg}"
    with open(RUN_LOG, "a") as f:
        f.write(line + "\n")
    print(line)

def normalize_sorting_object(obj, sampling_frequency):
    if not isinstance(obj, dict):
        raise RuntimeError("sorting_object is not a dict")
    if "firings" in obj:
        return {
            "sorting_format": obj.get("sorting_format", "mda"),
            "data": {
                "firings": obj["firings"],
                "samplerate": obj.get("samplerate", obj.get("sampling_frequency", sampling_frequency)),
            },
        }
    return obj

def load_sorting_from_object(obj, sampling_frequency):
    so = normalize_sorting_object(obj, sampling_frequency)
    sformat = str(so.get("sorting_format", "mda")).lower()
    data = so.get("data", {})
    if not isinstance(data, dict):
        raise RuntimeError("sorting_object.data invalid")
    firings_uri = data.get("firings") or data.get("firingsUri")
    if not firings_uri:
        raise RuntimeError("sorting_object missing firings")
    firings_path = kc.load_file(firings_uri)
    if firings_path is None:
        raise RuntimeError(f"Could not load firings uri: {firings_uri}")
    if sformat == "mda":
        sfreq = float(data.get("samplerate", data.get("sampling_frequency", sampling_frequency)))
        return sie.MdaSortingExtractor(firings_path, sampling_frequency=sfreq)
    if sformat == "npz":
        return sie.NpzSortingExtractor(firings_path)
    raise RuntimeError(f"Unsupported sorting_format: {sformat}")

def _unit_candidates(uid):
    out = [uid]
    try:
        s = str(uid)
        if s not in out:
            out.append(s)
    except Exception:
        pass
    try:
        i = int(uid)
        if i not in out:
            out.append(i)
    except Exception:
        pass
    return out

def _spike_train_robust(sorting, uid, seg=0):
    for c in _unit_candidates(uid):
        try:
            st = sorting.get_unit_spike_train(c, segment_index=seg)
            return np.asarray(st, dtype=np.int64).ravel(), True
        except Exception:
            pass
    return np.array([], dtype=np.int64), False

def to_int_sorting(sorting):
    uids = list(sorting.get_unit_ids())
    nseg = int(sorting.get_num_segments())
    sfreq = float(sorting.get_sampling_frequency())
    times_list, labels_list = [], []
    map_new_to_old = {}
    failures = 0
    for seg in range(nseg):
        seg_t, seg_l = [], []
        for new_id, old_id in enumerate(uids):
            st, ok = _spike_train_robust(sorting, old_id, seg=seg)
            if not ok:
                failures += 1
                continue
            if st.size == 0:
                continue
            map_new_to_old[new_id] = old_id
            seg_t.append(st)
            seg_l.append(np.full(st.shape, new_id, dtype=np.int64))
        if seg_t:
            tt = np.concatenate(seg_t)
            ll = np.concatenate(seg_l)
            order = np.argsort(tt, kind="stable")
            times_list.append(tt[order])
            labels_list.append(ll[order])
        else:
            times_list.append(np.array([], dtype=np.int64))
            labels_list.append(np.array([], dtype=np.int64))
    if failures > max(10, int(0.2 * max(len(uids), 1))):
        raise RuntimeError(f"Too many unit ID lookup failures while canonicalizing: {failures}")
    rebuilt = si.NumpySorting.from_times_labels(times_list=times_list, labels_list=labels_list, sampling_frequency=sfreq)
    return rebuilt, map_new_to_old

def load_recording(study_name, recording_name, recordings_uri):
    try:
        return sf.load_spikeforest_recording(study_name=study_name, recording_name=recording_name, uri=recordings_uri)
    except TypeError:
        return sf.load_spikeforest_recording(study_name=study_name, recording_name=recording_name)

def make_key(row):
    return f"{row['study_set']}/{row['study_name']}/{row['recording_name']}/{row['sorter_name']}"

if MANIFEST_JSON.exists():
    manifest_rows = json.loads(MANIFEST_JSON.read_text())
else:
    raise RuntimeError("Run manifest enumeration cell first")

if len(manifest_rows) == 0:
    raise RuntimeError(
        f"Manifest has 0 rows at {MANIFEST_JSON}. Re-run enumeration and relax STUDY_SETS/ALLOWED_SORTERS filters."
    )

state = {
    "started_at": datetime.now().isoformat(),
    "completed_keys": [],
    "failed_keys": {},
    "heartbeat_at": None,
    "current_key": None,
}
if STATE_JSON.exists():
    prev = json.loads(STATE_JSON.read_text())
    state.update(prev)

completed = set(state.get("completed_keys", []))
failed = dict(state.get("failed_keys", {}))
preflight = []

def categorize_error(exc):
    msg = str(exc).lower()
    if "sorting_object" in msg:
        return "sorting_object_invalid"
    if "unit id" in msg or "canonicaliz" in msg:
        return "unit_id_canonicalization_failed"
    if "compare_sorter_to_ground_truth" in msg or "ufunc 'equal'" in msg:
        return "compare_failed"
    if "recording not found" in msg:
        return "recording_not_found"
    if "timeout" in msg or "gateway" in msg or "connection" in msg:
        return "network_timeout"
    return "other"

for i, row in enumerate(manifest_rows, start=1):
    key = make_key(row)
    part_path = PARTS_DIR / f"{key.replace('/', '__')}.parquet"

    state["heartbeat_at"] = datetime.now().isoformat()
    state["current_key"] = key
    STATE_JSON.write_text(json.dumps(state, indent=2))

    if key in completed and part_path.exists():
        continue

    try:
        R = load_recording(row["study_name"], row["recording_name"], row["source_recordings_uri"])
        recording = R.get_recording_extractor()
        sorting_gt = R.get_sorting_true_extractor()
        sorting_out = load_sorting_from_object(row["sorting_object"], recording.get_sampling_frequency())

        sorting_gt_i, _ = to_int_sorting(sorting_gt)
        sorting_out_i, out_map = to_int_sorting(sorting_out)

        cmp = sc.compare_sorter_to_ground_truth(
            sorting_gt_i,
            sorting_out_i,
            exhaustive_gt=True,
            match_score=0.5,
            chance_score=0.1,
        )
        perf = cmp.get_performance(method="by_unit", output="dataframe").copy()
        perf.index = [int(x) for x in perf.index]

        fs = float(recording.get_sampling_frequency())
        n_samples = int(recording.get_num_samples())
        duration_sec = n_samples / max(fs, 1.0)
        n_channels = int(recording.get_num_channels())

        out_rows = []
        for uid in sorting_out_i.get_unit_ids():
            uid = int(uid)
            if uid not in perf.index:
                continue
            st = np.asarray(sorting_out_i.get_unit_spike_train(uid, segment_index=0), dtype=np.int64)
            n_spikes = int(st.size)
            rate_hz = float(n_spikes / max(duration_sec, 1e-9))
            isi_sec = np.diff(st) / fs if n_spikes > 1 else np.array([], dtype=np.float64)
            isi_mean_ms = float(np.mean(isi_sec) * 1000.0) if isi_sec.size > 0 else np.nan
            isi_cv = float(np.std(isi_sec) / np.mean(isi_sec)) if isi_sec.size > 1 and np.mean(isi_sec) > 0 else np.nan
            isi_viol_ratio = float(np.mean(isi_sec < 0.0015)) if isi_sec.size > 0 else 0.0

            p = perf.loc[uid]
            recall = float(p.get("recall", np.nan))
            precision = float(p.get("precision", np.nan))
            accuracy = float(p.get("accuracy", np.nan))

            original_uid = str(out_map.get(uid, uid))
            group_key = f"{row['study_set']}::{row['study_name']}::{row['recording_name']}"

            out_rows.append({
                "study_set": row["study_set"],
                "study_name": row["study_name"],
                "recording_name": row["recording_name"],
                "sorter_name": row["sorter_name"],
                "unit_id": original_uid,
                "group_key": group_key,
                "row_uid": f"{group_key}::{row['sorter_name']}::{original_uid}",
                "rec_n_channels": n_channels,
                "rec_sampling_frequency": fs,
                "rec_duration_sec": duration_sec,
                "unit_n_spikes": n_spikes,
                "unit_rate_hz": rate_hz,
                "unit_isi_mean_ms": isi_mean_ms,
                "unit_isi_cv": isi_cv,
                "unit_isi_viol_ratio": isi_viol_ratio,
                "fmiss": 1.0 - recall if np.isfinite(recall) else np.nan,
                "fpos": 1.0 - precision if np.isfinite(precision) else np.nan,
                "accuracy": accuracy,
            })

        if not out_rows:
            raise RuntimeError("No comparable units produced for this row")

        pd.DataFrame(out_rows).to_parquet(part_path, index=False)
        completed.add(key)
        failed.pop(key, None)
        preflight.append({"key": key, "status": "ready", "n_units": len(out_rows), "error": "", "category": "ok"})

        del recording, sorting_gt, sorting_out, sorting_gt_i, sorting_out_i
        gc.collect()
    except Exception as exc:
        failed[key] = str(exc)
        preflight.append({"key": key, "status": "skip", "n_units": 0, "error": str(exc), "category": categorize_error(exc)})

    state["completed_keys"] = sorted(completed)
    state["failed_keys"] = failed
    state["heartbeat_at"] = datetime.now().isoformat()
    STATE_JSON.write_text(json.dumps(state, indent=2))

    if i % 20 == 0:
        log(f"Progress {i}/{len(manifest_rows)} | completed={len(completed)} failed={len(failed)}")

preflight_df = pd.DataFrame(preflight)
preflight_rows_path = ARTIFACTS_DIR / "preflight_rows.parquet"
if not preflight_df.empty:
    preflight_df.to_parquet(preflight_rows_path, index=False)
failed_categories = (
    preflight_df.loc[preflight_df["status"] == "skip", "category"].value_counts().to_dict()
    if (not preflight_df.empty and "category" in preflight_df.columns)
    else {}
)

part_files = sorted(PARTS_DIR.glob("*.parquet"))
if not part_files:
    summary = {
        "generated_at": datetime.now().isoformat(),
        "n_manifest_rows": len(manifest_rows),
        "n_completed": len(completed),
        "n_failed": len(failed),
        "n_train_rows": 0,
        "failed_categories": failed_categories,
        "sample_failed_keys": list(failed.keys())[:30],
        "top_failed": dict(pd.Series(list(failed.values())).value_counts().head(20)) if failed else {},
        "preflight_rows_path": str(preflight_rows_path),
        "state_path": str(STATE_JSON),
    }
    PREFLIGHT_JSON.write_text(json.dumps(summary, indent=2))
    raise RuntimeError(
        "No feature parts were produced. "
        f"Inspect {PREFLIGHT_JSON} and {STATE_JSON} for exact failure categories/keys."
    )

df_train = pd.concat([pd.read_parquet(p) for p in part_files], ignore_index=True)
df_train = df_train.drop_duplicates(subset=["study_set", "study_name", "recording_name", "sorter_name", "unit_id"]).reset_index(drop=True)
df_train.to_parquet(TRAIN_PARQUET, index=False)

summary = {
    "generated_at": datetime.now().isoformat(),
    "n_manifest_rows": len(manifest_rows),
    "n_completed": len(completed),
    "n_failed": len(failed),
    "n_train_rows": len(df_train),
    "failed_categories": failed_categories,
    "top_failed": dict(pd.Series(list(failed.values())).value_counts().head(20)) if failed else {},
    "preflight_rows_path": str(preflight_rows_path),
}
PREFLIGHT_JSON.write_text(json.dumps(summary, indent=2))

print("Extraction finished")
print(json.dumps(summary, indent=2))



RuntimeError: No feature parts were produced. Inspect /Users/paulruiz/Documents/Predicting_Good_Units/local_debug/artifacts/preflight_summary.json and /Users/paulruiz/Documents/Predicting_Good_Units/local_debug/artifacts/extract_state.json for exact failure categories/keys.

In [ ]:
import json, pandas as pd
from pathlib import Path

root = Path("/Users/paulruiz/Documents/Predicting_Good_Units/local_debug/artifacts")
print(json.loads((root/"preflight_summary.json").read_text()))
df = pd.read_parquet(root/"preflight_rows.parquet")
print(df["category"].value_counts(dropna=False).head(20))
display(df[df["status"]=="skip"][["key","category","error"]].head(30))


{'generated_at': '2026-03-07T19:41:38.397839', 'n_manifest_rows': 0, 'n_completed': 0, 'n_failed': 0, 'n_train_rows': 0, 'failed_categories': {}, 'sample_failed_keys': [], 'top_failed': {}, 'preflight_rows_path': '/Users/paulruiz/Documents/Predicting_Good_Units/local_debug/artifacts/preflight_rows.parquet', 'state_path': '/Users/paulruiz/Documents/Predicting_Good_Units/local_debug/artifacts/extract_state.json'}


FileNotFoundError: [Errno 2] No such file or directory: '/Users/paulruiz/Documents/Predicting_Good_Units/local_debug/artifacts/preflight_rows.parquet'

In [ ]:
from scipy.stats import spearmanr
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, r2_score
import xgboost as xgb

if not TRAIN_PARQUET.exists():
    raise RuntimeError("Training parquet not found. Run extraction cell first.")

df = pd.read_parquet(TRAIN_PARQUET)

meta_cols = ["study_set", "study_name", "recording_name", "sorter_name", "unit_id", "group_key", "row_uid"]
target_cols = ["fmiss", "fpos", "accuracy"]
feat_cols = [c for c in df.columns if c not in meta_cols + target_cols]

if not feat_cols:
    raise RuntimeError("No feature columns found")

audit = {
    "generated_at": datetime.now().isoformat(),
    "n_rows": int(len(df)),
    "n_groups": int(df["group_key"].nunique()),
    "n_recordings": int(df["recording_name"].nunique()),
    "n_sorters": int(df["sorter_name"].nunique()),
    "n_features": int(len(feat_cols)),
    "duplicate_unit_rows": int(df.duplicated(subset=["study_set", "study_name", "recording_name", "sorter_name", "unit_id"]).sum()),
}
AUDIT_JSON.write_text(json.dumps(audit, indent=2))
print(json.dumps(audit, indent=2))

# grouped holdout (avoids recording leakage)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
groups = df["group_key"].astype(str).values
idx_tr, idx_te = next(gss.split(df[feat_cols], groups=groups))

df_tr = df.iloc[idx_tr].copy()
df_te = df.iloc[idx_te].copy()
medians = df_tr[feat_cols].median(numeric_only=True)

X_tr = df_tr[feat_cols].apply(pd.to_numeric, errors="coerce").fillna(medians).values.astype(np.float32)
X_te = df_te[feat_cols].apply(pd.to_numeric, errors="coerce").fillna(medians).values.astype(np.float32)

results = {}
pred_export = df_te[meta_cols + target_cols].copy()

for target in ["fmiss", "fpos"]:
    y_tr = df_tr[target].values.astype(np.float32)
    y_te = df_te[target].values.astype(np.float32)

    model = xgb.XGBRegressor(
        tree_method="hist",
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=SEED,
        eval_metric="mae",
    )
    model.fit(X_tr, y_tr)
    pred = np.clip(model.predict(X_te), 0, 1)
    pred_export[f"pred_{target}"] = pred

    mae = float(mean_absolute_error(y_te, pred))
    r2 = float(r2_score(y_te, pred)) if len(y_te) > 1 else np.nan
    rho = float(spearmanr(y_te, pred).statistic) if len(y_te) > 1 else np.nan
    results[target] = {"mae": mae, "r2": r2, "spearman": rho, "n": int(len(y_te))}

    model_path = MODELS_DIR / f"model_{target}.json"
    model.save_model(str(model_path))

METRICS_JSON.write_text(json.dumps(results, indent=2))
pred_export.to_csv(PREDICTIONS_CSV, index=False)

print("Holdout metrics:")
print(json.dumps(results, indent=2))


RuntimeError: Training parquet not found. Run extraction cell first.

## Debug Checklist

- If extraction is too slow, reduce `MAX_ROWS`.
- If a key hangs, inspect `local_debug/artifacts/extract_state.json` (`current_key`, `heartbeat_at`).
- Failed keys are recorded in `failed_keys`; fix those categories first before scaling scope.
- Once stable here, port fixes back to the main Colab notebook.
